<a href="https://colab.research.google.com/github/lokeshp051/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lokeshp051/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
BASE = "hf://datasets/FlyRank/internship-warehouse"

In [10]:
base = con.sql(f"""
WITH daily AS (
  SELECT report_date, client_hash_id, content_hash_id,
         gsc_impressions, gsc_clicks, gsc_sum_position
  FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet')
  WHERE gsc_data_available IS TRUE
),
h1 AS (
  SELECT client_hash_id, content_hash_id,
         SUM(gsc_impressions) AS impressions_h1,
         SUM(gsc_clicks) AS clicks_h1,
         SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions),0) AS avg_position_h1,
         SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions),0) AS ctr_h1
  FROM daily WHERE report_date <= DATE '2026-03-15'
  GROUP BY 1,2
)
SELECT h1.*, d.content_updated_date,
       DATE_DIFF('day', d.content_updated_date, DATE '2026-03-15') AS days_since_update
FROM h1
LEFT JOIN read_parquet('{BASE}/dim_content.parquet') d USING (content_hash_id)
WHERE h1.impressions_h1 > 20
""").df()

base.shape

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(108769, 8)

In [11]:
import pandas as pd
base["staleness_bucket"] = pd.cut(base["days_since_update"], bins=[-1,30,90,180,99999],
                                    labels=["<30d","30-90d","90-180d",">180d"])
verdict_table = base.groupby("staleness_bucket").agg(
    n=("content_hash_id","count"),
    avg_ctr=("ctr_h1","mean"),
    avg_position=("avg_position_h1","mean")
)
print(verdict_table)

                      n   avg_ctr  avg_position
staleness_bucket                               
<30d              20003  0.002146     14.291931
30-90d               98  0.002658     30.532770
90-180d             171  0.005470     22.851372
>180d                24  0.001024     13.738306


/tmp/ipykernel_3482/3113003333.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  verdict_table = base.groupby("staleness_bucket").agg(


In [12]:
base["position_bucket"] = pd.cut(base["avg_position_h1"], bins=[0,5,10,20,9999],
                                   labels=["top5","top10","top20",">20"])
ctr_table = base.groupby("position_bucket").agg(
    n=("content_hash_id","count"),
    avg_ctr=("ctr_h1","mean")
)
print(ctr_table)

                     n   avg_ctr
position_bucket                 
top5             30122  0.004019
top10            32767  0.003195
top20            19475  0.002675
>20              26397  0.001375


/tmp/ipykernel_3482/3747862346.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ctr_table = base.groupby("position_bucket").agg(


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Two signals checked:

Staleness (n=20,003/98/171/24 across buckets) → MIXED verdict. CTR and position don't degrade cleanly with staleness in this slice — the pattern is inconsistent, so I'm not using staleness in my rule. This negative result is useful: it stopped me from building on a weak assumption.
CTR-vs-position (n=30,122/32,767/19,475/26,397) → CONFIRMED verdict. CTR drops cleanly and monotonically as position worsens (0.0040 → 0.0014), matching the logic behind FlyRank's CTR-fix flag.

My rule: flag any page ranking well (top 20 or better) whose CTR is below the median CTR for its own position bucket — meaning it's under-clicking relative to peers at the same rank, a clear CTR-fix opportunity.
Reason code: LOW_CTR_FOR_POSITION
Action label: review_title_and_snippet

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [13]:
import numpy as np
import os

median_ctr_by_bucket = base.groupby("position_bucket")["ctr_h1"].median()
base["median_ctr_for_bucket"] = base["position_bucket"].map(median_ctr_by_bucket)

base["action_score"] = (base["median_ctr_for_bucket"] - base["ctr_h1"]).clip(lower=0)
base["reason_code"] = np.where(
    (base["position_bucket"].isin(["top5","top10","top20"])) & (base["ctr_h1"] < base["median_ctr_for_bucket"]),
    "LOW_CTR_FOR_POSITION", "no_flag"
)
base["action_label"] = np.where(base["reason_code"]=="LOW_CTR_FOR_POSITION",
                                  "review_title_and_snippet", "no_action")

queue = base[base["reason_code"]=="LOW_CTR_FOR_POSITION"].sort_values(
    ["action_score","impressions_h1"], ascending=[False, False])

os.makedirs("work/outputs", exist_ok=True)
queue[["client_hash_id","content_hash_id","position_bucket","ctr_h1",
       "median_ctr_for_bucket","action_score","reason_code","action_label"]].to_csv(
    "work/outputs/baseline_action_score.csv", index=False)

print("Queue size:", len(queue))
queue.head(10)

/tmp/ipykernel_3482/988789618.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  median_ctr_by_bucket = base.groupby("position_bucket")["ctr_h1"].median()


Queue size: 15061


,client_hash_id,content_hash_id,impressions_h1,clicks_h1,avg_position_h1,ctr_h1,content_updated_date,days_since_update,staleness_bucket,position_bucket,median_ctr_for_bucket,action_score,reason_code,action_label
30057,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,83772.0,0.0,0.105274,0.0,2026-02-25,18,<30d,top5,0.001881,0.001881,LOW_CTR_FOR_POSITION,review_title_and_snippet
61519,client_62f4a7e64f5e0096,content_0c5606abaaab3178,27715.0,0.0,4.448854,0.0,2026-07-04,-111,NaN,top5,0.001881,0.001881,LOW_CTR_FOR_POSITION,review_title_and_snippet
66000,client_1a730cb2640a1abf,content_d61fc394d10cba41,18554.0,0.0,2.992886,0.0,2026-05-25,-71,NaN,top5,0.001881,0.001881,LOW_CTR_FOR_POSITION,review_title_and_snippet
93004,client_62f4a7e64f5e0096,content_8d395745c4d76f9e,15012.0,0.0,4.410605,0.0,2026-07-03,-110,NaN,top5,0.001881,0.001881,LOW_CTR_FOR_POSITION,review_title_and_snippet
100683,client_fef1a8f436438636,content_66bf45eb0c5bb550,11457.0,0.0,1.793227,0.0,2026-06-17,-94,NaN,top5,0.001881,0.001881,LOW_CTR_FOR_POSITION,review_title_and_snippet
61547,client_62f4a7e64f5e0096,content_2d3ea336a4467aa5,9307.0,0.0,4.340067,0.0,2026-07-04,-111,NaN,top5,0.001881,0.001881,LOW_CTR_FOR_POSITION,review_title_and_snippet
85796,client_73cda7b4e4f265ea,content_41d18608d90de375,8569.0,0.0,1.344031,0.0,2026-02-25,18,<30d,top5,0.001881,0.001881,LOW_CTR_FOR_POSITION,review_title_and_snippet
20769,client_62f4a7e64f5e0096,content_82391bdebfa94156,8560.0,0.0,4.401636,0.0,2026-07-03,-110,NaN,top5,0.001881,0.001881,LOW_CTR_FOR_POSITION,review_title_and_snippet
76085,client_62f4a7e64f5e0096,content_03f33581fac04b10,8308.0,0.0,2.022749,0.0,2026-07-03,-110,NaN,top5,0.001881,0.001881,LOW_CTR_FOR_POSITION,review_title_and_snippet
100689,client_fef1a8f436438636,content_551a522809e0358c,7379.0,0.0,4.707277,0.0,2026-06-17,-94,NaN,top5,0.001881,0.001881,LOW_CTR_FOR_POSITION,review_title_and_snippet


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

1. content_0c057b66... — 83,772 impressions, position 0.11, CTR 0.0 → review_title_and_snippet. Why: ranks #1 essentially, but zero clicks — huge gap. Wrong if: query is branded/navigational (users already know the URL, skip clicking naturally).
2. content_0c5608ab... — 27,715 impressions, position 4.45, CTR 0.0 → same action. Why: strong position, no clicks. Wrong if: this is a duplicate/redirect page search engines index but users never actually land on.
3. content_d61fc394... — 18,554 impressions, position 2.99, CTR 0.0 → same. Wrong if: title tag is already optimized and the real issue is a technical rendering bug hiding the snippet.
4. content_8d395745... — 15,012 impressions, position 4.41, CTR 0.0 → same. Wrong if: SERP feature (e.g. featured snippet from another source) is stealing all clicks regardless of title quality.
5. content_66bf45eb... — 11,457 impressions, position 1.79, CTR 0.0 → same. Wrong if: impressions are inflated by bot/crawler traffic, not real users.
6. content_2d3ea336... — 9,307 impressions, position 4.34, CTR 0.0 → same. Wrong if: this content overlaps heavily with a sibling page also ranking, splitting clicks elsewhere.
7. content_41d18068... — 8,569 impressions, position 1.34, CTR 0.0 → same. Wrong if: query intent doesn't match page content, so users skip it even at top position — a content mismatch, not a snippet problem.
8. content_82391bde... — 8,560 impressions, position 4.40, CTR 0.0 → same. Wrong if: seasonal/temporary drop in interest for the query itself, unrelated to page quality.
9. content_03f33581... — 8,308 impressions, position 2.02, CTR 0.0 → same. Wrong if: this is a very recent page still accumulating an initial click lag.
10. content_551a5228... — 7,379 impressions, position 4.71, CTR 0.0 → same. Wrong if: measurement noise — a small time window where clicks haven't synced yet.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

All ten picks share ctr_h1 = 0.0 exactly — worth flagging as a weak pattern: exact-zero CTR at high impression counts is unusual and may indicate a data artifact (e.g. click tracking gap) rather than a genuine content problem for all ten. No future-window or label-derived inputs were used — everything comes from impressions_h1, clicks_h1, avg_position_h1 (first-half-of-month only), so this rule is safe from leakage.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.